# 2A · Tarea A: intención

Práctica final · **Asistente de agenda en lenguaje natural** · Procesamiento del Lenguaje Natural

Poned en la misma carpeta este notebook, `utilidades.py`, `evaluar.py`, `train.jsonl` y `agenda.jsonl`. Las celdas marcadas con `# TODO` son las que tenéis que completar.

Cada enfoque es una función `predecir(textos) -> lista de intenciones`. Las plantillas devuelven `"otro"` para que el notebook se ejecute desde el principio.

In [1]:
import time, json, os, io, itertools, warnings
import joblib
import pandas as pd
from utilidades import cargar, guardar, particiones, INTENCIONES, TIPOS, HOY
from evaluar import evaluar, imprimir
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
import tarea_a
from tarea_a import crear_modelo, PREPRO, DEFECTO

warnings.filterwarnings("ignore", category=UserWarning)
os.makedirs("modelos", exist_ok=True); os.makedirs("resultados", exist_ok=True)
pd.set_option("display.max_colwidth", 100)

train, val = particiones()
agenda = cargar("agenda.jsonl")
X_train, X_val = [x["texto"] for x in train], [x["texto"] for x in val]
y_train = [x["intencion"] for x in train]
RESULTADOS = {}      # enfoque -> métricas y predicciones en validación
MODELOS = {}         # enfoque -> modelo entrenado (para exportar el elegido a tarea_a.py)

def registrar(nombre, preds, segundos, tamano=None, datos=None, interpretable=None):
    """tamano: bytes del modelo; datos: frases de entrenamiento; interpretable: sí / parcial / no (columnas de la tabla del notebook 3)."""
    r = evaluar(val, preds)
    rd = evaluar(val, preds, "dificil") if any(x["dificil"] for x in val) else {}
    rn = evaluar(val, preds, "normal")
    RESULTADOS[nombre] = {"metrica": r["A_f1_macro"], "normales": rn["A_f1_macro"], "dificiles": rd.get("A_f1_macro"),
                          "accuracy": r["A_accuracy"], "ms_por_frase": 1000 * segundos / max(len(preds), 1),
                          "tamano_bytes": tamano, "datos_entrenamiento": datos, "interpretable": interpretable, "preds": preds}
    print(f"{nombre:<40} A_f1_macro = {r['A_f1_macro']:.3f}   normales = {rn['A_f1_macro']:.3f}   difíciles = {rd.get('A_f1_macro', '—'):.3f}"
          f"   {RESULTADOS[nombre]['ms_por_frase']:.2f} ms/frase")

def tamano(modelo):
    """Bytes del modelo serializado con joblib (lo que ocuparía en modelos/)."""
    b = io.BytesIO(); joblib.dump(modelo, b); return b.getbuffer().nbytes

# Validación cruzada estratificada SOLO sobre mi_train para elegir hiperparámetros (la validación no se toca)
CV = StratifiedKFold(5, shuffle=True, random_state=0)
def buscar_hiperparametros(enfoque, rejilla, top=5):
    """Prueba todas las combinaciones de la rejilla con validación cruzada en mi_train y devuelve la mejor."""
    filas = []
    for valores in itertools.product(*rejilla.values()):
        p = dict(zip(rejilla, valores))
        s = cross_val_score(crear_modelo(enfoque, **p), X_train, y_train, cv=CV, scoring="f1_macro")
        filas.append((s.mean(), s.std(), p))
    filas.sort(key=lambda f: -f[0])
    display(pd.DataFrame([{**{k: str(v) for k, v in p.items()}, "F1 CV (media)": m, "desv.": d}
                          for m, d, p in filas[:top]]).round(3))
    return filas[0][2]

def entrenar_y_registrar(nombre, enfoque, params, interpretable):
    modelo = crear_modelo(enfoque, **params).fit(X_train, y_train)
    modelo.predict(X_val[:1])                                   # calentamiento (carga de spaCy, etc.)
    t0 = time.time(); pred = list(modelo.predict(X_val)); seg = time.time() - t0
    registrar(nombre, preds_A(pred), seg, tamano(modelo), len(X_train), interpretable)
    MODELOS[nombre] = modelo
    return modelo

In [2]:
def preds_A(etiquetas):
    return [{"id": x["id"], "intencion": e} for x, e in zip(val, etiquetas)]

### A1 · Reglas
Palabras clave y expresiones regulares (temas 1–2).

Las reglas están en `tarea_a.py` (`intencion_reglas`) y se han escrito **mirando solo `mi_train.jsonl`**. Se aplican sobre el texto en minúsculas, sin tildes y con las abreviaturas de mensajería expandidas (`normalizar`), en este **orden de prioridad**:

1. `otro` si hay un indicio claro de fuera de dominio que se parece al dominio (el tiempo, fútbol, «¿qué hora es?», «recomiéndame»…), porque esas frases llevan fechas y «cuándo».
2. `borrar`: raíces *borr-, elimin-, quit-, cancel-, anul-*, «olvídate de», «se ha cancelado/suspendido». Antes se quitan las **negaciones** («no borres el dentista, solo muévelo» no es borrar).
3. `completar`: «ya …» (sin «ya no»), «he + participio», participios de hecho (*pagada, devueltos, terminada*; pero no «tener hecho»), *tacha*, «márcalo como hecho».
4. `modificar`: *cambi-, muev-, retras-, adelant-, aplaz-*, «lo han pasado al…», «ahora es», «no es el X, es el Y». Antes se protege «pasado mañana».
5. `buscar`: pregunta por un dato de una entrada (*cuándo, dónde, a qué hora, qué día, hasta cuándo*, «tenía», «era»).
6. `consultar`: pregunta por un día o periodo («qué tengo», «tengo algo», «estoy libre», «enséñame», «resúmeme»).
7. `crear_evento` / `crear_tarea`: si hay orden de apuntar o marcas de creación, es evento cuando hay **hora** o un sustantivo de evento sin marcas de tarea (*reunión, cita, cena, examen…*), y tarea si hay marcas de tarea (*tengo que, hay que, recuérdame, antes del, urgente, para el 20*…).
8. Si no encaja nada, `otro`.

Las órdenes explícitas de apuntar («apunta», «recuérdame»…) desactivan las reglas 3–6, para que «Recuérdame que tengo que…» no acabe en `consultar`.

In [3]:
from tarea_a import intencion_reglas
pred_tr = [intencion_reglas(t) for t in X_train]
print(f"F1 macro en mi_train (donde se diseñaron las reglas): {f1_score(y_train, pred_tr, average='macro'):.3f}")
t0 = time.time(); pred = [intencion_reglas(t) for t in X_val]; seg = time.time() - t0
reglas = {k: v for k, v in vars(tarea_a).items() if k.startswith("R_")}
print(f"{len(reglas)} expresiones regulares, {sum(len(r.pattern) for r in reglas.values())} caracteres en total")
registrar("A1 reglas", preds_A(pred), seg, sum(len(r.pattern.encode()) for r in reglas.values()), 0, "sí")

F1 macro en mi_train (donde se diseñaron las reglas): 0.991
14 expresiones regulares, 2498 caracteres en total
A1 reglas                                A_f1_macro = 0.908   normales = 0.953   difíciles = 0.845   0.04 ms/frase


### A2 · Bolsa de palabras + Naive Bayes
Tema 4.

`CountVectorizer` + `MultinomialNB`. Hiperparámetros elegidos con **validación cruzada de 5 particiones sobre `mi_train`**: preprocesado (las configuraciones de E2), unigramas o uni+bigramas y el suavizado `alpha`.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
mejor_A2 = buscar_hiperparametros("A2", dict(prepro=["original", "normalizado", "sin_stopwords", "stemming"],
                                             ngramas=[(1, 1), (1, 2)], alpha=[0.1, 0.3, 1.0]))
print("Elegido:", mejor_A2)
m = entrenar_y_registrar("A2 BoW + Naive Bayes", "A2", mejor_A2, "sí")
print("Vocabulario:", len(m.named_steps["bow"].vocabulary_), "palabras · parámetros:", m.named_steps["nb"].feature_log_prob_.size)

,prepro,ngramas,alpha,F1 CV (media),desv.
0,normalizado,"(1, 1)",0.1,0.690,0.043
1,normalizado,"(1, 1)",0.3,0.685,0.041
2,normalizado,"(1, 2)",0.3,0.675,0.045
3,normalizado,"(1, 2)",1.0,0.670,0.035
4,normalizado,"(1, 2)",0.1,0.653,0.051


Elegido: {'prepro': 'normalizado', 'ngramas': (1, 1), 'alpha': 0.1}
A2 BoW + Naive Bayes                     A_f1_macro = 0.713   normales = 0.776   difíciles = 0.639   0.03 ms/frase
Vocabulario: 472 palabras · parámetros: 3776


In [5]:
# Interpretabilidad: las palabras con más peso (log P(w|clase) relativo a la media) en cada intención
import numpy as np
nb, voc = m.named_steps["nb"], np.array(m.named_steps["bow"].get_feature_names_out())
rel = nb.feature_log_prob_ - nb.feature_log_prob_.mean(axis=0)
pd.DataFrame({c: voc[np.argsort(-rel[i])[:8]] for i, c in enumerate(nb.classes_)}).T

,0,1,2,3,4,5,6,7
borrar,borra,quita,elimina,cancela,suspendido,olvidate,ha,se
buscar,cuando,era,tenia,donde,hasta,curriculum,dia,acuerdo
completar,ya,he,esta,terminado,tacha,renovado,puse,marca
consultar,algo,libre,dime,tareas,esta,estoy,fin,planes
crear_evento,9,en,media,apunta,11,6,cuarto,febrero
crear_tarea,antes,recuerdame,comprar,prisa,urgente,apuntes,corre,sin
modificar,cambia,ahora,retrasa,mueve,pasa,movido,han,1201
otro,vale,gracias,cuanto,tiempo,quien,un,hacer,estoy


### A3 · TF-IDF + regresión logística
Palabras y n-gramas de caracteres (tema 4).

`FeatureUnion` de TF-IDF de palabras y de **n-gramas de caracteres** (`char_wb`), que comparten subcadenas entre variantes con faltas o abreviaturas («peluqeria» / «peluquería», «bórralo» / «borra»). Se eligen por validación cruzada el preprocesado, los n-gramas y la regularización `C`.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
mejor_A3 = buscar_hiperparametros("A3", dict(prepro=["original", "normalizado"], ngramas=[(1, 1), (1, 2)],
                                             chars=[(2, 4), (2, 5)], C=[1.0, 10.0, 100.0]))
print("Elegido:", mejor_A3)
m = entrenar_y_registrar("A3 TF-IDF + RL", "A3", mejor_A3, "parcial")
print("Rasgos:", m.named_steps["rl"].coef_.shape[1], "· parámetros:", m.named_steps["rl"].coef_.size + m.named_steps["rl"].intercept_.size)

,prepro,ngramas,chars,C,F1 CV (media),desv.
0,normalizado,"(1, 2)","(2, 4)",10.0,0.844,0.020
1,normalizado,"(1, 1)","(2, 5)",100.0,0.842,0.035
2,normalizado,"(1, 1)","(2, 4)",100.0,0.841,0.033
3,normalizado,"(1, 1)","(2, 4)",10.0,0.832,0.038
4,normalizado,"(1, 2)","(2, 4)",100.0,0.831,0.014


Elegido: {'prepro': 'normalizado', 'ngramas': (1, 2), 'chars': (2, 4), 'C': 10.0}
A3 TF-IDF + RL                           A_f1_macro = 0.896   normales = 0.961   difíciles = 0.808   0.06 ms/frase
Rasgos: 4645 · parámetros: 37168


### A4 · Word2Vec propio (media de vectores) + clasificador
Entrenadlo con gensim sobre vuestro corpus (tema 4).

Word2Vec *skip-gram* entrenado con gensim **solo sobre las frases de `mi_train`** (las de validación no, para no filtrar información), con `workers=1` y semilla fija para que sea reproducible. Cada frase se representa con la media de los vectores de sus palabras y se clasifica con regresión logística. Con ~300 frases y ~3 000 palabras, el corpus es diminuto para Word2Vec: la mayoría de las palabras salen una o dos veces, así que sus vectores apenas se mueven de la inicialización aleatoria.

Los avisos `Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'` que salen durante la búsqueda vienen del código C de gensim con algunas combinaciones de la rejilla (50 dimensiones y 200 épocas); no aparecen con la configuración elegida.

In [7]:
# from gensim.models import Word2Vec   (dentro de tarea_a.MediaWord2Vec)
mejor_A4 = buscar_hiperparametros("A4", dict(prepro=["normalizado"], dim=[50, 100], epocas=[50, 200], C=[0.1, 1.0]))
print("Elegido:", mejor_A4)
m = entrenar_y_registrar("A4 Word2Vec propio + clasificador", "A4", mejor_A4, "no")
wv = m.named_steps["w2v"].w2v_
print(f"Vocabulario Word2Vec: {len(wv)} palabras × {wv.vector_size} dim.")
for w in ["borra", "reunion", "mañana", "cuando"]:
    if w in wv: print(f"  más parecidas a «{w}»:", [p for p, _ in wv.most_similar(w, topn=5)])

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


,prepro,dim,epocas,C,F1 CV (media),desv.
0,normalizado,100,50,1.0,0.664,0.039
1,normalizado,100,200,1.0,0.652,0.039
2,normalizado,50,200,0.1,0.644,0.017
3,normalizado,100,200,0.1,0.637,0.031
4,normalizado,50,200,1.0,0.630,0.041


Elegido: {'prepro': 'normalizado', 'dim': 100, 'epocas': 50, 'C': 1.0}
A4 Word2Vec propio + clasificador        A_f1_macro = 0.607   normales = 0.619   difíciles = 0.581   0.04 ms/frase
Vocabulario Word2Vec: 473 palabras × 100 dim.
  más parecidas a «borra»: ['han', 'juego', 'gano', 'suspendido', 'compra']
  más parecidas a «reunion»: ['grupo', 'datos', 'bases', 'martes', 'elena']
  más parecidas a «mañana»: ['empieza', 'cine', 'mas', 'voy', 'tarde']
  más parecidas a «cuando»: ['matricula', 'jurafsky', 'pagado', 'lo', 'me']


### A5 · Embeddings estáticos preentrenados
Vectores de spaCy o FastText en español.

Media de los vectores de 300 dimensiones de `es_core_news_md` (entrenados por spaCy sobre un corpus grande en español) + regresión logística. Como los vectores son de palabras con tildes, aquí el preprocesado candidato «abreviaturas» expande la mensajería sin quitar tildes. También se prueba a promediar solo las palabras de contenido (sin stopwords, salvo *no, ya, qué, cuándo*).

In [8]:
mejor_A5 = buscar_hiperparametros("A5", dict(prepro=["original", "normalizado", "abreviaturas"], solo_contenido=[False, True],
                                             C=[0.01, 0.1, 1.0]))
print("Elegido:", mejor_A5)
m = entrenar_y_registrar("A5 vectores preentrenados + clasificador", "A5", mejor_A5, "no")
from utilidades import _nlp
vec = _nlp().vocab.vectors
RESULTADOS["A5 vectores preentrenados + clasificador"]["tamano_bytes"] += vec.data.nbytes   # + la tabla de vectores de spaCy
print(f"Vectores de spaCy: {vec.shape[0]} vectores únicos × {vec.shape[1]} dim. = {vec.data.nbytes / 1e6:.0f} MB")

,prepro,solo_contenido,C,F1 CV (media),desv.
0,abreviaturas,False,0.1,0.764,0.059
1,abreviaturas,False,1.0,0.754,0.056
2,normalizado,False,0.1,0.739,0.040
3,normalizado,False,1.0,0.731,0.044
4,abreviaturas,False,0.01,0.721,0.035


Elegido: {'prepro': 'abreviaturas', 'solo_contenido': False, 'C': 0.1}
A5 vectores preentrenados + clasificador A_f1_macro = 0.843   normales = 0.889   difíciles = 0.771   0.28 ms/frase
Vectores de spaCy: 20000 vectores únicos × 300 dim. = 24 MB


### A6 · Embeddings de frase + regresión logística
El modelo **no se entrena**: calculad los vectores una vez y reutilizadlos en A7 (tema 4, parte 2).

In [9]:
# Fase 3: A6 embeddings de frase + RL (se rellena tras el tema 4, parte 2)
# from sentence_transformers import SentenceTransformer
# m = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# E_train, E_val = m.encode(X_train), m.encode(X_val)

### A7 · Los mismos embeddings + Random Forest y MLP
¿Qué clasificador aprovecha mejor los embeddings?

In [10]:
# Fase 3: A7 mismos embeddings + Random Forest y MLP

### A8 · LLM sin entrenar: zero-shot y few-shot
Convertid la respuesta en una de las 8 intenciones y contad las inválidas. Usad al menos un modelo de 1,5B.

In [11]:
# Fase 3: A8 LLM zero-shot y few-shot (Qwen2.5-1.5B-Instruct, ejemplos few-shot de mi_train)

> Para los experimentos E1 y E2 (notebook 3), añadid también a `tarea_a.py` una función `entrenar_y_predecir(enfoque, textos_train, etiquetas_train, textos_val)` para A2, A3, A6 y A7.

## Resumen y enfoque elegido
Guardamos los resultados de validación para el notebook 3 y elegimos un enfoque.

In [12]:
tabla_A = pd.DataFrame({k: {c: v[c] for c in ("metrica", "normales", "dificiles", "accuracy", "ms_por_frase", "tamano_bytes")}
                        for k, v in RESULTADOS.items()}).T
tabla_A["tamano_KB"] = tabla_A.pop("tamano_bytes") / 1024
display(tabla_A.astype(float).round(3))
json.dump(RESULTADOS, open("resultados/resultados_A.json", "w", encoding="utf-8"), ensure_ascii=False)

,metrica,normales,dificiles,accuracy,ms_por_frase,tamano_KB
A1 reglas,0.908,0.953,0.845,0.909,0.036,2.441
A2 BoW + Naive Bayes,0.713,0.776,0.639,0.727,0.025,66.091
A3 TF-IDF + RL,0.896,0.961,0.808,0.896,0.058,379.355
A4 Word2Vec propio + clasificador,0.607,0.619,0.581,0.623,0.035,205.352
A5 vectores preentrenados + clasificador,0.843,0.889,0.771,0.844,0.284,23455.743


**Comentario (fase 2: A1–A5).**
* **Las reglas (A1) son el mejor enfoque de la fase 2** (F1 macro 0,908) y el más rápido y pequeño, pero hay que leerlo con cuidado: en `mi_train`, donde se escribieron, sacan 0,991, así que casi una de cada diez frases nuevas ya se les escapa. Fallan con lo que no se anticipó al escribirlas: paráfrasis sin verbo de acción («Fuera lo del basket»), «pásalo a…» y una negación con completado («Ya no tengo que mandar el currículum, lo mandé ayer»). Cada regla nueva arregla un caso pero puede romper otro, y su mantenimiento crece con el número de reglas.
* **A3 (TF-IDF de palabras + n-gramas de caracteres) casi empata con las reglas** (0,896) sin escribir ninguna regla, y es el mejor en frases normales (0,961). Los n-gramas de caracteres son los que marcan la diferencia con A2: comparten subcadenas entre *bórralo/borra*, *peluqeria/peluquería* o *mñn/mañana*.
* **A2 (BoW + Naive Bayes) se queda en 0,713**: con 308 frases, muchas palabras discriminantes salen una o dos veces y NB no tiene manera de generalizar a las no vistas; confunde sobre todo `otro` y `consultar`, que comparten fechas y «qué».
* **A4 (Word2Vec propio) es el peor (0,607)**: con ~3 000 palabras de entrenamiento los vectores no aprenden semántica (las vecinas de «borra» son *han, juego, gano*), y la media de vectores diluye los verbos de acción entre las palabras de relleno.
* **A5 (vectores de spaCy) llega a 0,843** con el mismo clasificador que A4: los vectores preentrenados sí traen semántica de un corpus grande. A cambio, el modelo pesa 23 MB (la tabla de vectores) y es el más lento (0,28 ms/frase, por el *pipeline* de spaCy).
* **Frases difíciles**: todos los enfoques bajan entre 4 y 15 puntos. Las reglas son las más robustas en difíciles (0,845) porque normalizan abreviaturas antes de buscar los patrones; A3 baja a 0,808. El desglose por tipo de dificultad es el experimento E3 (notebook 3, fase 5).

**Elección provisional.** Exportamos el mejor enfoque **entrenable** (A3) y no las reglas, aunque A1 saque 0,012 más en validación: la diferencia es de una frase de 77, A3 no depende de que alguien haya previsto cada formulación y se reentrena sin esfuerzo al añadir frases. La elección definitiva se hace en la fase 3, cuando estén A6–A8.

In [13]:
# Elección PROVISIONAL (fase 2): el mejor enfoque entrenable de A1–A5. Se revisa en la fase 3 con A6–A8.
candidatos = [k for k in RESULTADOS if k in MODELOS]
MEJOR = max(candidatos, key=lambda k: RESULTADOS[k]["metrica"])
print("Enfoque elegido (provisional):", MEJOR)
joblib.dump(MODELOS[MEJOR], tarea_a.RUTA_MODELO)
print(f"Guardado en {os.path.relpath(tarea_a.RUTA_MODELO)} ({os.path.getsize(tarea_a.RUTA_MODELO) / 1024:.0f} KB)")

Enfoque elegido (provisional): A3 TF-IDF + RL
Guardado en modelos/tarea_a.joblib (379 KB)


## Exportar el enfoque elegido
Copiad a **`tarea_a.py`** el enfoque elegido (ya entrenado o con lo necesario para entrenarlo) como una función `predecir_intencion(texto) -> intención`. La usan el asistente (notebook 4) y la función `generar_predicciones` del notebook 3, que es la que ejecutará el profesor con el **test oculto**.

`tarea_a.predecir_intencion` carga el modelo guardado en `modelos/tarea_a.joblib` (el pipeline completo, preprocesado incluido). Comprobamos que funciona aislado y que da lo mismo que el modelo en memoria:

In [14]:
import importlib; importlib.reload(tarea_a)
p = [tarea_a.predecir_intencion(t) for t in X_val]
assert p == [x["intencion"] for x in RESULTADOS[MEJOR]["preds"]]
for t in ["q tngo mñn??", "Quita lo de la peluquería", "¿Qué tiempo hace mañana?", "El viernes ceno con Andrea a las 9"]:
    print(f"{t:<40} → {tarea_a.predecir_intencion(t)}")

q tngo mñn??                             → consultar
Quita lo de la peluquería                → borrar
¿Qué tiempo hace mañana?                 → otro
El viernes ceno con Andrea a las 9       → crear_evento
